# 第1幕『縮める』 — 量子化を測る

> **シリーズの物語** 「小さなモデルを、三つの角度から解剖する」
>
> 同じ小型モデル（Qwen2.5-0.5B など）を主役にして、3冊を通して一本の筋で学びます。
>
> 1. **第1幕『縮める』** — 量子化。重みの精度を落とすと、サイズ・速度・賢さがどう動くかを測る。
> 2. **第2幕『方向を知る』** — abliteration 論文の読解。「拒否は1つの方向に乗っている」という主張を、概念として理解する（コードで外す作業はしない）。
> 3. **第3幕『中を覗く』** — 解釈性。第2幕の土台である「内部表現を読む」技術を、安全な題材（事実の想起）で自分の手で動かす。
>
> 縮めて速くし、その中で何が起きているかを知り、最後に内部を直接観察する——という流れです。


## このノートで分かること

大きな言語モデルは重みが FP16（1個あたり16ビット）で保存されています。
**量子化**は、この数値を4ビットや2ビットなど粗い表現に置き換えて、サイズと必要メモリを削る技術です。

問いは1つ: **「どこまで削ると、どれだけ賢さを失うのか？」**

このノートでは小さなモデルを FP16 / INT8 / INT4 で読み込み、
- モデルサイズ（ディスク・メモリ）
- 推論速度（tokens/sec）
- 言語モデルとしての質（perplexity = 次の単語をどれだけ当てられるか。低いほど良い）

を**実際に測って表とグラフにします**。GGUF の Q2/Q4/Q8 も考え方は同じです。


## 動かす環境について

> このノートは**あなたのPC**で動かす前提です（GPU があると速いですが、0.5B なら CPU でも動きます）。
> クラウドのこの環境には PyTorch も外部ダウンロードもないため、ここでは実行できません。

必要なもの:
```bash
pip install torch transformers bitsandbytes accelerate datasets matplotlib
```
`bitsandbytes` は INT8/INT4 量子化に使います（NVIDIA GPU 前提。Mac の場合は後述の GGUF 節へ）。


In [ ]:
import time, torch, gc
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B"   # 小さくて素直。0.5B なら気軽に試せる
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

tok = AutoTokenizer.from_pretrained(MODEL)


## ステップ1 — 3つの精度でモデルを読み込む関数

同じモデルを「読み込み方」だけ変えて3種類用意します。
- **fp16**: 元の精度（基準）
- **int8**: `load_in_8bit`
- **int4**: `load_in_4bit`（NF4 という4ビット形式）


In [ ]:
from transformers import BitsAndBytesConfig

def load(kind):
    if kind == "fp16":
        return AutoModelForCausalLM.from_pretrained(
            MODEL, torch_dtype=torch.float16).to(DEVICE)
    if kind == "int8":
        return AutoModelForCausalLM.from_pretrained(
            MODEL, quantization_config=BitsAndBytesConfig(load_in_8bit=True))
    if kind == "int4":
        return AutoModelForCausalLM.from_pretrained(
            MODEL, quantization_config=BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16))
    raise ValueError(kind)

def mem_mb(model):
    # パラメータ+バッファの実バイト数を合計
    b = sum(p.numel()*p.element_size() for p in model.parameters())
    b += sum(x.numel()*x.element_size() for x in model.buffers())
    return b/1e6


## ステップ2 — 速度を測る

同じプロンプトから一定トークン生成し、1秒あたり何トークン出せたかを測ります。


In [ ]:
def gen_speed(model, prompt="The history of science is", n_new=64):
    ids = tok(prompt, return_tensors="pt").to(model.device)
    torch.cuda.synchronize() if DEVICE=="cuda" else None
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=n_new, do_sample=False)
    torch.cuda.synchronize() if DEVICE=="cuda" else None
    dt = time.time()-t0
    made = out.shape[1]-ids["input_ids"].shape[1]
    return made/dt, tok.decode(out[0], skip_special_tokens=True)


## ステップ3 — 質を測る（perplexity）

perplexity は「モデルがテキストをどれだけ自然だと思っているか」の指標です。
テスト文を食わせ、各位置での予測確率から計算します。**低いほど良い**。
量子化で情報を削ると、普通はここが少しずつ悪化（上昇）します。


In [ ]:
# 評価用テキスト（手早く試すため短め。本格的には wikitext-2 の test を使う）
EVAL_TEXT = (
    "Machine learning models learn patterns from data. "
    "Quantization reduces the numerical precision of model weights "
    "to save memory and increase inference speed, usually at a small "
    "cost in accuracy. Understanding this trade-off is essential."
) * 8

@torch.no_grad()
def perplexity(model, text=EVAL_TEXT, stride=256):
    ids = tok(text, return_tensors="pt").input_ids.to(model.device)
    max_len = getattr(model.config, "max_position_embeddings", 1024)
    nll, ntok = 0.0, 0
    for i in range(0, ids.size(1), stride):
        chunk = ids[:, i:i+max_len]
        if chunk.size(1) < 2: break
        out = model(chunk, labels=chunk)
        nll += out.loss.item()*(chunk.size(1)-1)
        ntok += chunk.size(1)-1
    return float(np.exp(nll/max(ntok,1)))


## ステップ4 — 3種類を順に測って表にする

In [ ]:
rows = []
for kind in ["fp16", "int8", "int4"]:
    try:
        m = load(kind)
        size = mem_mb(m)
        spd, sample = gen_speed(m)
        ppl = perplexity(m)
        rows.append(dict(kind=kind, size_MB=round(size,1),
                         tok_per_s=round(spd,1), perplexity=round(ppl,2)))
        print(f"[{kind}] {size:.0f}MB  {spd:.1f} tok/s  ppl={ppl:.2f}")
        print("   sample:", sample[:90].replace(chr(10)," "), "...")
    except Exception as e:
        print(f"[{kind}] skip:", e)
    finally:
        del m; gc.collect()
        torch.cuda.empty_cache() if DEVICE=="cuda" else None


In [ ]:
import pandas as pd
df = pd.DataFrame(rows).set_index("kind")
# fp16 を基準に相対サイズ・相対劣化を出す
base = df.loc["fp16"]
df["size_vs_fp16"] = (df.size_MB/base.size_MB).round(2)
df["ppl_increase_%"] = ((df.perplexity/base.perplexity-1)*100).round(1)
df


## ステップ5 — グラフで『トレードオフ』を見る

左ほど小さい、下ほど賢い。右上に外れるほど『削りすぎ』のサイン。

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,2, figsize=(11,4))
ax[0].bar(df.index, df.size_MB); ax[0].set_title("Model size (MB)"); ax[0].set_ylabel("MB")
ax[1].plot(df.size_MB, df.perplexity, "o-")
for k,r in df.iterrows(): ax[1].annotate(k,(r.size_MB,r.perplexity))
ax[1].set_xlabel("size (MB) → 小さいほど良い")
ax[1].set_ylabel("perplexity → 低いほど良い")
ax[1].set_title("サイズ vs 質 のトレードオフ")
plt.tight_layout(); plt.show()


## 発展: GGUF（Ollama / llama.cpp）側で同じことをする

`bitsandbytes` は NVIDIA GPU 向けです。Mac や Ollama ユーザーは **GGUF の Q2_K / Q4_K_M / Q8_0** で同じ比較ができます。
考え方（サイズ↓・速度↑・ppl↑のトレードオフ）は完全に同じです。

```bash
# 量子化ごとに perplexity を測る（wikitext などのテキストを渡す）
llama-perplexity -m model-Q2_K.gguf   -f wiki.test.raw
llama-perplexity -m model-Q4_K_M.gguf -f wiki.test.raw
llama-perplexity -m model-Q8_0.gguf   -f wiki.test.raw
# 速度は実行時の eval rate を見る
ollama run <model>:Q4_K_M "..." --verbose
```


## まとめ（第1幕）

- 量子化は **サイズ・メモリを大きく削れる**が、**perplexity はじわじわ悪化**する。
- 多くの場合 **Q4〜INT8 あたりが「ほぼ無劣化で軽い」おいしい帯**。Q2 まで落とすと劣化が目立ち始める。
- 「どこまで削るか」はいつも**トレードオフの選択**。数値で見ると判断できる。

> **次の幕へ** — 量子化は「重みの数値をどう表すか」の話でした。
> では、モデルの中で **意味（たとえば『断る』という振る舞い）はどう表されている**のでしょう？
> 第2幕では、それを論文の主張として読み解きます。
